# Analisis Data Emosi Teks Bahasa Indonesia

Dataset: `data/dataset_clean_ataraxia.csv` (kalimat berlabel 8 emosi). Notebook ini memuat data, memvalidasi kualitas, eksplorasi, melatih model, dan mengevaluasinya. Model, pembagian data, dan normalisasi teks sama dengan `app.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
DATA_PATH = "data/dataset_clean_ataraxia.csv"
LABELS = ["senang", "sedih", "marah", "takut", "kaget", "jijik", "cape", "netral"]

def clean(text):
    return " ".join(str(text).lower().split())

## 1. Memuat data

In [ ]:
df = pd.read_csv(DATA_PATH)
df["text"] = df["text"].map(clean)
print("Jumlah baris:", len(df))
df.head()

## 2. Validasi kualitas data

In [ ]:
pd.Series({
    "baris": len(df),
    "nilai kosong": int(df.isna().sum().sum()),
    "baris duplikat": int(df.duplicated().sum()),
    "teks duplikat": int(df["text"].duplicated().sum()),
    "label di luar 8 kelas": int((~df["label"].isin(LABELS)).sum()),
    "teks kosong": int((df["text"] == "").sum()),
})

Tidak ada nilai kosong, duplikat, maupun label di luar delapan kelas, sehingga data siap dipakai.

## 3. Eksplorasi data

In [ ]:
df["jumlah_kata"] = df["text"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, y="label", order=LABELS, ax=ax[0], color="#4C72B0")
ax[0].set_title("Distribusi label")
sns.histplot(df["jumlah_kata"], bins=30, ax=ax[1], color="#55A868")
ax[1].set_title("Panjang teks (jumlah kata)")
plt.tight_layout(); plt.show()

df["jumlah_kata"].describe().round(2)

In [ ]:
for lab in LABELS:
    print(f"[{lab}]")
    for t in df[df["label"] == lab]["text"].sample(2, random_state=42):
        print("  -", t)

Setiap label berjumlah sama (2.000 data) dan teksnya berupa kalimat pendek (median sekitar 7 kata).

## 4. Model dan evaluasi

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                      LogisticRegression(max_iter=2000, C=5))
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"Data latih: {len(X_train)} | Data uji: {len(X_test)}")
print(f"Akurasi: {accuracy_score(y_test, pred):.4f} | F1 makro: {f1_score(y_test, pred, average='macro'):.4f}\n")
print(classification_report(y_test, pred, labels=LABELS, digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=LABELS, cmap="Blues",
                                        colorbar=False, xticks_rotation=45)
plt.title("Confusion matrix (data uji)"); plt.tight_layout(); plt.show()

In [ ]:
salah = pd.DataFrame({"teks": X_test.values, "label_asli": y_test.values, "prediksi": pred})
salah = salah[salah["label_asli"] != salah["prediksi"]]
print("Jumlah salah prediksi:", len(salah), "dari", len(X_test))
salah.head(10)

## 5. Contoh prediksi

In [ ]:
contoh = ["uangku habis dan aku marah.",
          "tidak ada konser band favorit di kota saya. aku kaget sekali!",
          "lunglai dan lelah. aku begadang semalam."]
for t in contoh:
    p = model.predict_proba([clean(t)])[0]
    print(f"{t}\n  -> {model.classes_[p.argmax()]} ({p.max():.1%})")

## Kesimpulan

- Data bersih dan seimbang; tidak diperlukan pembersihan tambahan.
- Model TF-IDF dan Logistic Regression mencapai akurasi sekitar 98% pada data uji.
- Kalimat pada dataset berpola seragam, sehingga akurasi tinggi belum menjamin performa yang sama pada teks bergaya berbeda. Hal itu belum diuji pada notebook ini.